In [1]:
import numpy as np
import pandas as pd


# -----------------------------
# Scenario settings
# -----------------------------
E_SIZE = 180
P_SIZE = 200
B_SIZE = 0
STORAGE_SIZE = 200
COMPRESSOR_CAPACITY = 8
INITIAL_STORAGE_STATUS = 0

ELECTROLYZER_EFFICIENCY = 62.7  # %
SELL_STORAGE_THRESHOLD = 0.95 * STORAGE_SIZE

H2_PRICES = [1.5]
FREE_DAY_RANGE = [3]

SIMULATION_YEARS = 14
PROJECT_LIFETIME = 30
DISCOUNT_RATE = 0.05


# -----------------------------
# Load data
# -----------------------------
PV = pd.read_csv("PVhourlyHydrogen500.csv", delimiter=";")
PriceAllYears = pd.read_csv("PriceAllYears.csv", delimiter=";")


# -----------------------------
# Helper values
# -----------------------------
discount_factor = sum(
    1 / (1 + DISCOUNT_RATE) ** t
    for t in range(1, PROJECT_LIFETIME + 1)
)

economic_df = pd.DataFrame()


# -----------------------------
# Simulation loop
# -----------------------------
for sell_hydrogen_price in H2_PRICES:
    for sell_free_days in FREE_DAY_RANGE:

        Simh = PV[
            ["Year", "Month", "Day", "Hour", "Production (Wh)", "Consumption (kWh H2"]
        ].copy()

        Simh["PV production (kWh)"] = (Simh["Production (Wh)"] / 1000) * P_SIZE
        Simh["Consumption (kg H2)"] = Simh["Consumption (kWh H2"] / 33.3

        # Mark hours where hydrogen selling is allowed
        lookahead_hours = sell_free_days * 24
        future_consumption = (
            Simh["Consumption (kg H2)"]
            .iloc[::-1]
            .rolling(lookahead_hours + 1, min_periods=1)
            .sum()
            .iloc[::-1]
        )

        Simh["Sell window"] = (future_consumption <= 1e-9).astype(int)

        # Add electricity prices
        Simh = Simh.merge(
            PriceAllYears,
            on=["Year", "Month", "Day", "Hour"],
            how="left"
        )

        price_threshold = Simh["N4 (EUR/kWh)"].quantile(0.8)
        storage_threshold = max(SELL_STORAGE_THRESHOLD, 0.9 * STORAGE_SIZE)

        battery_status = [0]
        storage_status = [INITIAL_STORAGE_STATUS]

        excess_energy = []
        electrolyzer_production = []
        pv_to_electrolyzer = []
        grid_to_electrolyzer = []
        compressor_flow_list = []
        hydrogen_sold = []
        h2_to_consumption_list = []
        h2_deficit_list = []
        sell_flag = []

        for i in range(len(Simh)):
            prev_battery = battery_status[-1]
            prev_storage = storage_status[-1]

            pv_production = Simh.loc[i, "PV production (kWh)"]
            current_price = Simh.loc[i, "N4 (EUR/kWh)"]
            consumption = Simh.loc[i, "Consumption (kg H2)"]

            # PV to electrolyzer
            storage_full = prev_storage >= STORAGE_SIZE - COMPRESSOR_CAPACITY
            compressor_full = i > 0 and compressor_flow_list[-1] >= COMPRESSOR_CAPACITY

            if compressor_full or storage_full:
                pv_to_el = 0
                excess_e = pv_production
            else:
                pv_to_el = min(pv_production, E_SIZE)
                excess_e = max(0, pv_production - pv_to_el)

            # Battery charging
            charged_battery = min(B_SIZE, prev_battery + excess_e)
            battery_charge_added = charged_battery - prev_battery
            excess_e = max(0, excess_e - battery_charge_added)

            # Battery to electrolyzer
            remaining_el_capacity = E_SIZE - pv_to_el
            battery_used = 0

            if remaining_el_capacity > 0 and prev_storage < STORAGE_SIZE - COMPRESSOR_CAPACITY:
                battery_used = min(charged_battery, remaining_el_capacity)

            new_battery = max(0, charged_battery - battery_used)

            # Grid to electrolyzer
            grid_input = 0
            remaining_el_capacity -= battery_used

            if (
                remaining_el_capacity > 0
                and current_price < price_threshold
                and prev_storage < min(storage_threshold, STORAGE_SIZE - COMPRESSOR_CAPACITY)
            ):
                grid_input = remaining_el_capacity

            available_energy = pv_to_el + battery_used + grid_input
            electrolyzer_energy = min(available_energy, E_SIZE)

            # Hydrogen production
            h2_produced = electrolyzer_energy * ELECTROLYZER_EFFICIENCY / 100 / 33.3
            compressor_flow = min(h2_produced, COMPRESSOR_CAPACITY)

            available_storage = min(
                STORAGE_SIZE,
                max(0, prev_storage + compressor_flow - consumption)
            )

            h2_to_consumption = min(consumption, available_storage)
            h2_deficit = consumption - h2_to_consumption

            # Hydrogen selling
            can_sell = (
                STORAGE_SIZE > 0
                and Simh.loc[i, "Sell window"] == 1
                and available_storage >= SELL_STORAGE_THRESHOLD
            )

            if can_sell:
                h2_sold = available_storage
                sell_event = 1
            else:
                h2_sold = 0
                sell_event = 0

            new_storage = min(
                STORAGE_SIZE,
                max(0, prev_storage + compressor_flow - consumption) - h2_sold
            )

            battery_status.append(new_battery)
            storage_status.append(new_storage)

            excess_energy.append(excess_e)
            electrolyzer_production.append(electrolyzer_energy)
            pv_to_electrolyzer.append(pv_to_el)
            grid_to_electrolyzer.append(grid_input)
            compressor_flow_list.append(compressor_flow)
            hydrogen_sold.append(h2_sold)
            h2_to_consumption_list.append(h2_to_consumption)
            h2_deficit_list.append(h2_deficit)
            sell_flag.append(sell_event)

        # Add simulation results
        Simh["Battery status (kWh)"] = battery_status[1:]
        Simh["Excess e (kWh)"] = excess_energy
        Simh["Electrolyzer production (kWh)"] = electrolyzer_production
        Simh["PV to Electrolyzer (kWh)"] = pv_to_electrolyzer
        Simh["Grid to Electrolyzer (kWh)"] = grid_to_electrolyzer
        Simh["Hydrogen produced (kg)"] = (
            Simh["Electrolyzer production (kWh)"]
            * ELECTROLYZER_EFFICIENCY
            / 100
            / 33.3
        )
        Simh["Compressor flow (kg H2)"] = compressor_flow_list
        Simh["H2 storage (kg H2)"] = storage_status[1:]
        Simh["Hydrogen sold (kg)"] = hydrogen_sold
        Simh["H2 to consumption (kg H2)"] = h2_to_consumption_list
        Simh["Selling event"] = sell_flag
        Simh["H2def (kg H2)"] = h2_deficit_list

        Simh.to_csv("Simh.csv", index=False, sep=";")

        # -----------------------------
        # Economic calculations
        # -----------------------------
        H2_Storage_Capex = 1100 * STORAGE_SIZE
        Annual_H2_Storage_Opex = 0.01 * H2_Storage_Capex

        Battery_Capex = 800 * B_SIZE
        Annual_Battery_Opex = 0.02 * Battery_Capex

        Compressor_Capex = 240000
        Annual_Compressor_Opex = 0.005 * Compressor_Capex

        PV_Capex = 900 * P_SIZE
        Annual_PV_Opex = 0.015 * PV_Capex

        Electrolyzer_Capex_unit = 1.1195 * (
            585.85 + (E_SIZE ** 0.622) * (9485.2 / E_SIZE)
        )
        Electrolyzer_CAPEX = E_SIZE * Electrolyzer_Capex_unit
        Annual_Electrolyzer_Opex = 0.02 * Electrolyzer_CAPEX
        Electrolyzer_Replacement_Cost = 0.35 * Electrolyzer_CAPEX

        Electrolyzer_Water_Consumption = 10  # L/kg H2
        Water_Cost = 0.00053  # EUR/L

        Monthly_N4_Fixed_Fee = 42
        Monthly_N4_Power_Fee = 4.1 * E_SIZE
        Annual_N4_Electricity_Fixed_Cost = 12 * (
            Monthly_N4_Fixed_Fee + Monthly_N4_Power_Fee
        )

        Total_CAPEX = (
            PV_Capex
            + Battery_Capex
            + Electrolyzer_CAPEX
            + H2_Storage_Capex
            + Compressor_Capex
        )

        Total_annual_OPEX_fix = (
            Annual_Electrolyzer_Opex
            + Annual_PV_Opex
            + Annual_Compressor_Opex
            + Annual_Battery_Opex
            + Annual_H2_Storage_Opex
            + Annual_N4_Electricity_Fixed_Cost
        )

        Total_Discounted_OPEX_fix = Total_annual_OPEX_fix * discount_factor

        Total_Discounted_Replacement_Cost = (
            Electrolyzer_Replacement_Cost / (1 + DISCOUNT_RATE) ** 10
            + Electrolyzer_Replacement_Cost / (1 + DISCOUNT_RATE) ** 20
        )

        Simh["Electricity Cost (EUR)"] = (
            Simh["Grid to Electrolyzer (kWh)"] * Simh["N4 (EUR/kWh)"]
        )
        Simh["Electricity Income (EUR)"] = (
            Simh["Excess e (kWh)"] * Simh["Income (EUR/kWh)"]
        )
        Simh["Hydrogen Sale Income (EUR)"] = (
            Simh["Hydrogen sold (kg)"] * sell_hydrogen_price
        )

        Annual_Electricity_Cost = Simh["Electricity Cost (EUR)"].sum() / SIMULATION_YEARS
        Annual_Electricity_Income = Simh["Electricity Income (EUR)"].sum() / SIMULATION_YEARS
        Annual_Hydrogen_Income = Simh["Hydrogen Sale Income (EUR)"].sum() / SIMULATION_YEARS

        Total_Discounted_Electricity_Cost = Annual_Electricity_Cost * discount_factor
        Total_Discounted_Electricity_Income = Annual_Electricity_Income * discount_factor
        Total_Discounted_Hydrogen_Income = Annual_Hydrogen_Income * discount_factor

        Annual_H2_Produced = Simh["Hydrogen produced (kg)"].sum() / SIMULATION_YEARS
        Annual_H2_Consumed = Simh["H2 to consumption (kg H2)"].sum() / SIMULATION_YEARS

        Annual_Water_Cost = (
            Electrolyzer_Water_Consumption
            * Water_Cost
            * Annual_H2_Produced
        )
        Total_Discounted_Water_Cost = Annual_Water_Cost * discount_factor

        Total_discounted_H2 = Annual_H2_Produced * discount_factor
        Total_discounted_H2_Consumption = Annual_H2_Consumed * discount_factor

        Net_Present_Cost = (
            Total_CAPEX
            + Total_Discounted_OPEX_fix
            + Total_Discounted_Replacement_Cost
            + Total_Discounted_Electricity_Cost
            + Total_Discounted_Water_Cost
            - Total_Discounted_Electricity_Income
            - Total_Discounted_Hydrogen_Income
        )

        LCOH = (
            Net_Present_Cost / Total_discounted_H2
            if Total_discounted_H2 > 0
            else np.nan
        )

        LCOHfarm = (
            Net_Present_Cost / Total_discounted_H2_Consumption
            if Total_discounted_H2_Consumption > 0
            else np.nan
        )

        economic_results = {
            "Storage threshold for sale": SELL_STORAGE_THRESHOLD,
            "No work ahead (days)": sell_free_days,
            "Hydrogen sell price (EURO/kg H2)": sell_hydrogen_price,
            "Discount Rate": DISCOUNT_RATE,
            "Battery Size (kWh)": B_SIZE,
            "Electrolyzer Size (kWp)": E_SIZE,
            "Storage Size (kg H2)": STORAGE_SIZE,
            "PV Size (kWp)": P_SIZE,
            "LCOH (EUR2024/kg H2)": LCOH,
            "LCOHfarm (EUR2024/kg H2)": LCOHfarm,
            "Net Present Cost (EUR2024)": Net_Present_Cost,
            "Total CAPEX (EUR2024)": Total_CAPEX,
            "PV Capex": PV_Capex,
            "Battery Capex": Battery_Capex,
            "Electrolyzer Capex": Electrolyzer_CAPEX,
            "H2 Storage Capex": H2_Storage_Capex,
            "Compressor Capex": Compressor_Capex,
            "Total Discounted Fixed OPEX (EUR2024)": Total_Discounted_OPEX_fix,
            "Annual Electrolyzer Opex": Annual_Electrolyzer_Opex,
            "Annual PV Opex": Annual_PV_Opex,
            "Annual Compressor Opex": Annual_Compressor_Opex,
            "Annual Battery Opex": Annual_Battery_Opex,
            "Annual H2 Storage Opex": Annual_H2_Storage_Opex,
            "Annual N4 Electricity Fixed Cost": Annual_N4_Electricity_Fixed_Cost,
            "Total Discounted Replacement Cost (EUR2024)": Total_Discounted_Replacement_Cost,
            "Total Discounted Electricity Cost (EUR2024)": Total_Discounted_Electricity_Cost,
            "Total Discounted Water Cost (EUR2024)": Total_Discounted_Water_Cost,
            "Total Discounted Electricity Income (EUR2024)": Total_Discounted_Electricity_Income,
            "Total Discounted Hydrogen Income (EUR2024)": Total_Discounted_Hydrogen_Income,
            "Total H2 produced (kg)": PROJECT_LIFETIME * Annual_H2_Produced,
            "Total H2 sold (kg)": PROJECT_LIFETIME * Simh["Hydrogen sold (kg)"].sum() / SIMULATION_YEARS,
            "Number of selling events": PROJECT_LIFETIME * Simh["Selling event"].sum() / SIMULATION_YEARS,
            "Total H2 deficit (kg)": PROJECT_LIFETIME * Simh["H2def (kg H2)"].sum() / SIMULATION_YEARS,
            "Electrolyzer usage (%)": 100 * Annual_H2_Produced * 33.3 / (8760 * E_SIZE * ELECTROLYZER_EFFICIENCY / 100),
            "Electrolyzer production (kWh)": PROJECT_LIFETIME * Simh["Electrolyzer production (kWh)"].sum() / SIMULATION_YEARS,
            "PV to Electrolyzer (kWh)": PROJECT_LIFETIME * Simh["PV to Electrolyzer (kWh)"].sum() / SIMULATION_YEARS,
            "Grid to Electrolyzer (kWh)": PROJECT_LIFETIME * Simh["Grid to Electrolyzer (kWh)"].sum() / SIMULATION_YEARS,
            "Consumption (kg)": PROJECT_LIFETIME * Simh["Consumption (kg H2)"].sum() / SIMULATION_YEARS,
            "PV production (kWh)": PROJECT_LIFETIME * Simh["PV production (kWh)"].sum() / SIMULATION_YEARS,
            "Excess electricity": PROJECT_LIFETIME * Simh["Excess e (kWh)"].sum() / SIMULATION_YEARS,
        }

        economic_df = pd.concat(
            [economic_df, pd.DataFrame([economic_results])],
            ignore_index=True
        )


economic_df.to_csv("economic_results.csv", index=False, sep=";")

In [2]:
economic_df

,Storage threshold for sale,No work ahead (days),Hydrogen sell price (EURO/kg H2),Discount Rate,Battery Size (kWh),Electrolyzer Size (kWp),Storage Size (kg H2),PV Size (kWp),LCOH (EUR2024/kg H2),LCOHfarm (EUR2024/kg H2),...,Total H2 sold (kg),Number of selling events,Total H2 deficit (kg),Electrolyzer usage (%),Electrolyzer production (kWh),PV to Electrolyzer (kWh),Grid to Electrolyzer (kWh),Consumption (kg),PV production (kWh),Excess electricity
0,190.0,3,1.5,0.05,0,180,200,200,7.112074,45.392494,...,553706.834217,2875.714286,0.0,73.739935,3.488194e+07,5285392.62,2.959655e+07,102904.92287,5.619717e+06,334324.821429
